# KALE + DPO with judge pairs (E5)

Starts from the retrained KALE checkpoint (`kale_retrain_colab.ipynb`) and runs DPO on our
LLM-as-a-judge preference pairs with `baselines/kale_dpo.py`. TRL's `DPOTrainer` can't train KALE
(it isn't a Hugging Face model), so the script implements the DPO loss on KALE's own model code.

1. Reference log-probs: score every chosen/rejected caption once with the frozen starting model.
2. DPO: only the text decoder trains; dropout off; the image encoder is frozen (like our Qwen LoRA runs).
   After each epoch it scores the 505 val pairs (DPO loss/accuracy) and the 253 val paintings (CIDEr),
   and keeps the best epoch by val CIDEr.
3. A few settings are tried; the best by val CIDEr is scored once on test.


## Settings

In [ ]:
SMOKE_TEST = True
DRIVE_DIR = "/content/drive/MyDrive/KALE_files"
SFT_CKPT_DRIVE = f'{DRIVE_DIR}/kale_sft_best{"_smoke" if SMOKE_TEST else ""}.pth'   # from kale_retrain_colab.ipynb
RESULTS_DRIVE = f'{DRIVE_DIR}/results_dpo{"_smoke" if SMOKE_TEST else ""}'

# Settings to try (name, beta, learning rate, epochs). Use E3's beta too if it differs, so the runs compare.
SWEEP = [
    ('b0.1_lr5e-6', 0.1, 5e-6, 3),
    ('b0.1_lr1e-6', 0.1, 1e-6, 3),
    ('b0.5_lr5e-6', 0.5, 5e-6, 3),
]
if SMOKE_TEST:
    SWEEP = [('smoke', 0.1, 5e-6, 1)]

## Mount Drive, get KALE and our repo, check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, glob, json, shutil, re
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists('/content/KALE'):
    !git clone -q https://github.com/Yanbei-Jiang/Artwork-Interpretation.git /content/KALE
# Our repo: kale_dpo.py and the judge files (train/val split + preference pairs)
if not os.path.exists('/content/emm'):
    !git clone -q https://github.com/emmadionne1/10623_Project_EMMbeddings.git /content/emm
else:
    !git -C /content/emm pull -q
# Quick fixes without pushing: put an edited kale_dpo.py in KALE_files/ on Drive and it is used instead
if os.path.exists(f'{DRIVE_DIR}/kale_dpo.py'):
    shutil.copy(f'{DRIVE_DIR}/kale_dpo.py', '/content/emm/baselines/kale_dpo.py'); print('using kale_dpo.py from Drive')
DPO = '/content/emm/baselines/kale_dpo.py'
assert os.path.exists(DPO), 'kale_dpo.py not found: push it to GitHub or copy it to KALE_files/ on Drive'
%cd /content/KALE
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
if 'L4' in gpu:
    print('WARNING: PyTorch 1.11 may not support the L4. Use an A100 (real runs) or a T4 (smoke test).')

## Python 3.9 environment for KALE

In [ ]:
PY = '/content/kale-env/bin/python'
!pip install -q uv
if not os.path.exists(PY):
    !uv venv /content/kale-env --python 3.9.16
!uv pip install -q --python {PY} torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!uv pip install -q --python {PY} numpy==1.23.5 "transformers==4.18.0" pyyaml ruamel.yaml wandb oss2 ftfy regex tqdm six more-itertools "opencv-python-headless<4.9" timm==0.4.12 gdown pycocoevalcap
!uv pip install -q --python {PY} git+https://github.com/bckim92/language-evaluation.git
!apt-get install -y -qq default-jre-headless > /dev/null
!MPLBACKEND=Agg {PY} -c "import language_evaluation; language_evaluation.download('coco'); print('language_evaluation ok')"
!{PY} -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0)); print(torch.ones(2).cuda()*2)"
RUNPY = f'MPLBACKEND=Agg WANDB_MODE=offline {PY}'   # prefix for every KALE-env command

## Data and image encoder

In [ ]:
assert os.path.getsize(f'{DRIVE_DIR}/ViT-L-14.tar') > 500e6, 'ViT-L-14.tar missing: run the replication notebook step 4c once'
!ln -sf {DRIVE_DIR}/ViT-L-14.tar /content/KALE/
!unzip -o -q $(find {DRIVE_DIR}/kale_shared -name data.zip | head -1) 'data/ArtCap/*' -d /content/KALE
if not os.path.exists('data/ArtCap/images'):
    !unzip -o -q $(find {DRIVE_DIR}/artcap_download -name images.zip | head -1) -d /content/KALE/data/ArtCap
train_val = json.load(open('data/ArtCap/artcap_train_val.json'))
test = json.load(open('data/ArtCap/artcap_test.json'))
missing = [e[0] for e in train_val + test if not os.path.exists(os.path.join('data/ArtCap/images', e[0]))]
print('train_val:', len(train_val), ' test:', len(test), ' (expected 2524 and 1082) | missing images:', len(missing))
assert not missing

# Our 90/10 split of KALE's train_val (seed 10623), recorded in the judge files
split_of = {str(json.loads(l)['image_id']): json.loads(l)['split'] for l in open('/content/emm/data/preferences/judgments.jsonl')}
img_id = lambda fn: os.path.splitext(os.path.basename(fn))[0]
train_entries = [e for e in train_val if split_of.get(img_id(e[0])) == 'train']
val_entries = [e for e in train_val if split_of.get(img_id(e[0])) == 'val']
print('our split: train', len(train_entries), ' val', len(val_entries), ' (expected 2271 and 253)')
assert len(train_entries) + len(val_entries) == len(train_val)

## Copy the starting checkpoint from Drive

In [ ]:
START = '/content/kale_sft_best.pth'
assert os.path.exists(SFT_CKPT_DRIVE), f'{SFT_CKPT_DRIVE} not found: run kale_retrain_colab.ipynb first'
if not (os.path.exists(START) and os.path.getsize(START) == os.path.getsize(SFT_CKPT_DRIVE)):
    shutil.copy(SFT_CKPT_DRIVE, START)
print(START, round(os.path.getsize(START) / 1e9, 2), 'GB')
os.makedirs(RESULTS_DRIVE, exist_ok=True)
SMOKE = '--smoke 32' if SMOKE_TEST else ''
BATCH = '--batch 16 --accum 2 --eval_batch 32' if 'A100' in gpu else '--batch 4 --accum 8 --eval_batch 8'

## Reference log-probs 
Look at the last line: the starting model should already give the chosen caption a higher probability
than the rejected one in somewhat more than half of the pairs.

In [ ]:
%cd /content/KALE
REF = f'/content/ref_logps{"_smoke" if SMOKE_TEST else ""}.pt'
REF_DRIVE = f'{RESULTS_DRIVE}/{os.path.basename(REF)}'
if os.path.exists(REF_DRIVE) and not os.path.exists(REF):
    shutil.copy(REF_DRIVE, REF)
if not os.path.exists(REF):
    !{RUNPY} {DPO} ref_logps --checkpoint {START} --out {REF} {BATCH} {SMOKE}
    shutil.copy(REF, REF_DRIVE)
else:
    print('reusing', REF)

## DPO runs

In [ ]:
%cd /content/KALE
for name, beta, lr, epochs in SWEEP:
    run_dir = f'/content/dpo_runs/{name}'
    if os.path.exists(f'{run_dir}/summary.json'):
        print('already done:', name); continue
    print(f'\n===== {name}: beta={beta} lr={lr} epochs={epochs}')
    !{RUNPY} {DPO} train --checkpoint {START} --ref {REF} --out_dir {run_dir} --beta {beta} --lr {lr} --epochs {epochs} {BATCH} {SMOKE}
    os.makedirs(f'{RESULTS_DRIVE}/{name}', exist_ok=True)
    for f in glob.glob(f'{run_dir}/*.json*'):
        shutil.copy(f, f'{RESULTS_DRIVE}/{name}/')

In [ ]:
import pandas as pd
rows = []
for name, *_ in SWEEP:
    for line in open(f'/content/dpo_runs/{name}/metrics.jsonl'):
        r = json.loads(line); r['run'] = name; rows.append(r)
runs = pd.DataFrame(rows)
cols = ['run', 'epoch', 'val_loss', 'val_acc', 'val_margin', 'val_logp_chosen', 'val_B1', 'val_B4', 'val_ROUGE_L', 'val_CIDEr', 'val_mean_len']
print(runs[cols].to_string(index=False))
runs[cols + ['example']].to_csv(f'{RESULTS_DRIVE}/dpo_val_by_epoch.csv', index=False)

summ = {name: json.load(open(f'/content/dpo_runs/{name}/summary.json')) for name, *_ in SWEEP}
BEST_RUN = max(summ, key=lambda n: summ[n]['best_val_CIDEr'])
print(f'\nbest run by val CIDEr: {BEST_RUN} (epoch {summ[BEST_RUN]["best_epoch"]}, '
      f'val CIDEr {summ[BEST_RUN]["best_val_CIDEr"]} vs {summ[BEST_RUN]["start_val_CIDEr"]} before DPO)')

## Score the chosen run on test
Run this once, after choosing the run on validation.

In [ ]:
!pip install -q pycocoevalcap
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
import pandas as pd

def score_file(path, meteor=False):
    """Score a KALE-format caption file against its own gold captions (x100), like helpers/metrics.py."""
    res = json.load(open(path))
    tok = PTBTokenizer()
    gen = tok.tokenize({i: [{'caption': ' '.join(str(r['pred_caption']).split()) or 'none'}] for i, r in enumerate(res)})
    ref = tok.tokenize({i: [{'caption': c} for c in r['gold_caption']] for i, r in enumerate(res)})
    out = {}
    b, _ = Bleu(4).compute_score(ref, gen, verbose=0)
    out.update({f'B{n}': v for n, v in enumerate(b, 1)})
    if meteor:
        try:
            m = Meteor().compute_score(ref, gen); out['METEOR'] = m[0] if isinstance(m, tuple) else m
        except Exception as e:
            print('METEOR failed (compute it later with helpers/metrics.py):', repr(e)[:120]); out['METEOR'] = float('nan')
    out['ROUGE_L'], _ = Rouge().compute_score(ref, gen)
    out['CIDEr'], _ = Cider().compute_score(ref, gen)
    out = {k: round(100 * v, 2) for k, v in out.items()}
    out['n'] = len(res); out['mean_len'] = round(sum(len(str(r['pred_caption']).split()) for r in res) / len(res), 1)
    return out

%cd /content/KALE
E5_OUT = f'/content/dpo_runs/{BEST_RUN}/test_captions.json'
!{RUNPY} {DPO} generate --checkpoint /content/dpo_runs/{BEST_RUN}/best.pth --split test --out {E5_OUT} {BATCH} {SMOKE}
e5 = score_file(E5_OUT, meteor=True)
shutil.copy(E5_OUT, f'{RESULTS_DRIVE}/{BEST_RUN}/')

E4_OUT = f'{DRIVE_DIR}/results_{"sft_smoke" if SMOKE_TEST else "sft_full"}/test_captions_sft.json'
table = {'E5: KALE + DPO (judge pairs)': e5}
if os.path.exists(E4_OUT):
    table = {'E4: KALE (retrained)': score_file(E4_OUT, meteor=True), **table}
else:
    print('E4 test captions not found at', E4_OUT)
table = pd.DataFrame(table).T
table.to_csv(f'{RESULTS_DRIVE}/test_E4_vs_E5.csv')
print(table.to_string())
ex = json.load(open(E5_OUT))[:5]
for r in ex: print(' ', r['image_id'], '->', r['pred_caption'])